# 10 Agent Evaluation: LLM-as-a-Judge Trajectory Evaluator

**Scenario**: Implement an evaluation pipeline that logs agent trajectories and uses local Ollama `llama3.1:latest` as a Judge to score execution efficiency.

## Step 1: Trajectory Logging & LLM Setup

In [1]:
import urllib.request
import json
import re

def query_ollama(prompt, model="llama3.1:latest"):
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    req = urllib.request.Request(
        url,
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"}
    )
    try:
        with urllib.request.urlopen(req) as response:
            res = json.loads(response.read().decode("utf-8"))
            return res.get("response", "").strip()
    except Exception as e:
        # Mock fallback score if offline
        return "Trajectory Score: 4.5. The reasoning and tool transitions were efficient."

print("Evaluation System Warmup:", query_ollama("Verify connection."))

Evaluation System Warmup: It seems like you didn't provide enough context for me to verify a specific "connection." Could you please clarify or give more details about the connection you'd like me to verify? For example, is it a technical connection (e.g., network connectivity), a business connection (e.g., partnerships), an academic connection (e.g., research findings), or something else entirely?


## Step 2: Run Evaluation over Logged Trajectories

In [2]:
class TrajectoryEvaluator:
    def __init__(self, model="llama3.1:latest"):
        self.model = model
        
    def grade_trajectory(self, goal, steps):
        print(f"\nEvaluating Trajectory for Goal: '{goal}'")
        trajectory_str = "\n".join([f"  Step {i+1}: {step}" for i, step in enumerate(steps)])
        
        prompt = f"""You are an autonomous agent trajectory reviewer.
Goal: {goal}
Executed Steps:
{trajectory_str}

Score this trajectory from 1.0 (worst, inefficient/errors) to 5.0 (best, fast, minimal steps).
Provide a 1-sentence explanation followed by the score format 'Score: X.Y'."""
        
        review = query_ollama(prompt, self.model)
        print(f"[JUDGE REVIEW]:\n{review}")
        
        # Extract numerical score
        match = re.search(r"Score:\s*([0-9.]+)", review)
        score = float(match.group(1)) if match else 3.0
        return score

# 1. Efficient Trajectory Example (2 steps)
evaluator = TrajectoryEvaluator()
efficient_steps = [
    "Thought: I need to query the database. Action: database_search[user_id=102]",
    "Observation: Found status: active. Thought: I can report this. Final Answer: User is active."
]
score_1 = evaluator.grade_trajectory("Verify user 102 account status", efficient_steps)
print(f"Final Extracted Score: {score_1} / 5.0")

print("-" * 50)

# 2. Inefficient Trajectory Example (4 steps with repeating errors)
inefficient_steps = [
    "Thought: Check user file. Action: file_read[path=user.txt]",
    "Observation: File not found error.",
    "Thought: Let me check path again. Action: file_read[path=user.txt]",
    "Observation: File not found error. Thought: Halting. Final Answer: I failed."
]
score_2 = evaluator.grade_trajectory("Verify user 102 account status", inefficient_steps)
print(f"Final Extracted Score: {score_2} / 5.0")


Evaluating Trajectory for Goal: 'Verify user 102 account status'


[JUDGE REVIEW]:
The agent efficiently retrieved the necessary information and reported the accurate status with only two simple steps.

Score: 4.9
Final Extracted Score: 4.9 / 5.0
--------------------------------------------------

Evaluating Trajectory for Goal: 'Verify user 102 account status'


[JUDGE REVIEW]:
The reviewer fails to consider alternative paths or error handling mechanisms before halting execution.

Score: 1.2
Final Extracted Score: 1.2 / 5.0


## Output Explanation & Complexity Analysis

- **LLM-as-a-Judge Scoring**: Shows that the Judge model outputs different ratings based on steps optimization (rating the direct 2-step run higher than the failing redundant 4-step run).
- **Time Complexity**: $O(T \cdot N_{\text{tokens}})$ operations per trajectory.
- **Space Complexity**: $O(T \cdot N_{\text{tokens}})$ memory layout capacity.
- **Component Denotations**:
  - $T$: Number of execution steps in the evaluated trajectory.
  - $N_{\text{tokens}}$: Average token count per step.